# Aula 6 — Testes de Hipótese na Regressão Múltipla

**Laboratórios de Econometria em R** · MPEF/FGV-EPGE · Prof. Marcelo Mello

---

Perguntas que envolvem **mais de um coeficiente** não se resolvem com uma estatística
$t$. A ferramenta é a **estatística $F$**. Cinco resultados:

1. a estatística de **Wald**, implementada do zero;
2. a identidade $F = t^2$ quando há uma restrição só;
3. a fórmula do $F$ via $R^2$ — e por que ela **não** deve ser usada;
4. o teste $F$ **homocedástico rejeita demais** sob heterocedasticidade (medido);
5. testar "um coeficiente de cada vez" **infla o tamanho** do teste (medido).

Qualquer conjunto de $q$ restrições lineares se escreve como
$\mathbf{R}\boldsymbol{\beta} = \mathbf{r}$, e a estatística é

$$F = \frac{1}{q}(\mathbf{R}\hat\beta - \mathbf{r})'\left[\mathbf{R}\,\widehat{\text{Var}}(\hat\beta)\,\mathbf{R}'\right]^{-1}(\mathbf{R}\hat\beta - \mathbf{r})$$

In [ ]:
for (p in c("ggplot2", "dplyr")) {
  if (!requireNamespace(p, quietly = TRUE)) install.packages(p, quiet = TRUE)
}
library(ggplot2)
suppressMessages(library(dplyr))
theme_set(theme_minimal(base_size = 13))
az <- "#1f4e79"; vd <- "#2e7d32"; vm <- "#b3261e"; cz <- "grey45"
options(repr.plot.width = 8, repr.plot.height = 4)

## 0. As funções: erro-padrão robusto e teste de Wald

Os pacotes `car` e `sandwich` fariam isso automaticamente, mas escrever a fórmula
deixa explícito o que o software faz por baixo do capô — e evita dependências.

In [ ]:
ep_robusto <- function(modelo, tipo = "HC1") {
  X <- model.matrix(modelo); u <- residuals(modelo)
  n <- nrow(X); k <- ncol(X)
  bread  <- solve(crossprod(X))
  ajuste <- if (tipo == "HC1") n / (n - k) else 1
  sqrt(diag(bread %*% (ajuste * crossprod(X * u)) %*% bread))
}

vcov_robusto <- function(modelo, tipo = "HC1") {
  X <- model.matrix(modelo); u <- residuals(modelo)
  n <- nrow(X); k <- ncol(X)
  bread  <- solve(crossprod(X))
  ajuste <- if (tipo == "HC1") n / (n - k) else 1
  bread %*% (ajuste * crossprod(X * u)) %*% bread
}

# teste F (Wald) para R %*% beta = r
teste_F <- function(modelo, R, r = rep(0, nrow(R)), robusto = TRUE) {
  b <- coef(modelo)
  V <- if (robusto) vcov_robusto(modelo) else vcov(modelo)
  q <- nrow(R); gl <- df.residual(modelo)
  dif <- R %*% b - r
  Fst <- as.numeric(t(dif) %*% solve(R %*% V %*% t(R)) %*% dif) / q
  list(F = Fst, q = q, gl = gl, p = pf(Fst, q, gl, lower.tail = FALSE))
}

tabela_reg <- function(modelo, robusto = TRUE) {
  b  <- coef(modelo)
  ep <- if (robusto) ep_robusto(modelo) else summary(modelo)$coefficients[, 2]
  t  <- b / ep; gl <- df.residual(modelo)
  data.frame(coef = b, ep = ep, t = t, p = 2 * pt(-abs(t), gl),
             ic_inf = b - qt(0.975, gl) * ep, ic_sup = b + qt(0.975, gl) * ep)
}

## 1. Os dados

Acrescentamos $Expn$ (despesa por aluno): correlacionada com $STR$ — distritos que
gastam mais têm turmas menores — **e** determinante da nota. As duas condições do viés
de variável omitida.

In [ ]:
set.seed(6205)
n_d <- 420
b0 <- 686.0; b1 <- -1.10; b2 <- -0.65; b3 <- 3.87

STR   <- rnorm(n_d, mean = 19.64, sd = 1.89)
rho   <- 0.19; sd_el <- 18.3
PctEL <- pmax(0, 35.8 + rho * (sd_el / 1.89) * (STR - 19.64) +
                 rnorm(n_d, 0, sd_el * sqrt(1 - rho^2)))
Expn  <- 5.31 - 0.20 * (STR - 19.64) + rnorm(n_d, 0, 0.80)
u     <- rnorm(n_d, 0, 14.4)
TestScore <- b0 + b1 * STR + b2 * PctEL + b3 * Expn + u

ca <- data.frame(TestScore, STR, PctEL, Expn)
round(cor(ca[, c("STR", "PctEL", "Expn")]), 3)

In [ ]:
m2 <- lm(TestScore ~ STR + PctEL, data = ca)          # omite Expn
m3 <- lm(TestScore ~ STR + PctEL + Expn, data = ca)   # completo

rbind(sem_Expn = c(coef(m2)[2], t = coef(m2)[2] / ep_robusto(m2)[2]),
      com_Expn = c(coef(m3)[2], t = coef(m3)[2] / ep_robusto(m3)[2])) |> round(4)

Ao incluir $Expn$, o coeficiente do $STR$ **encolhe** em direção ao valor verdadeiro
($-1{,}10$) e a estatística $t$ despenca — quase perdendo significância. Duas coisas
se movem: o viés foi removido (ganho) **e** o erro-padrão cresceu por
multicolinearidade (perda).

## 2. A identidade $F = t^2$

In [ ]:
f1 <- teste_F(m2, matrix(c(0, 1, 0), nrow = 1))   # H0: beta_STR = 0
t1 <- unname(coef(m2)[2] / ep_robusto(m2)[2])

c(t = t1, t_ao_quadrado = t1^2, F = f1$F, p = f1$p) |> round(6)

Coincidem até o último dígito. Com $q=1$ o teste $F$ **é** o quadrado do teste $t$ —
razão pela qual o valor crítico da $F_{1,\infty}$ a 5% é $3{,}84 = 1{,}96^2$.

## 3. O teste conjunto, por três caminhos

$H_0: \beta_{STR} = 0$ **e** $\beta_{Expn} = 0$ — duas restrições.

In [ ]:
R_conj <- rbind(c(0, 1, 0, 0),    # beta_STR  = 0
                c(0, 0, 0, 1))    # beta_Expn = 0

f_rob <- teste_F(m3, R_conj, robusto = TRUE)
f_hom <- teste_F(m3, R_conj, robusto = FALSE)

# a fórmula via R² (só vale sob homocedasticidade)
m_restrito <- lm(TestScore ~ PctEL, data = ca)
R2_u <- summary(m3)$r.squared; R2_r <- summary(m_restrito)$r.squared
q <- 2; k_u <- 3
F_r2 <- ((R2_u - R2_r) / q) / ((1 - R2_u) / (n_d - k_u - 1))

c(F_robusto = f_rob$F, F_homocedastico = f_hom$F, F_pela_formula_R2 = F_r2) |> round(4)

As duas últimas coincidem: a fórmula via $R^2$ e a forma de Wald com a matriz
convencional são **a mesma estatística**, escrita de dois modos. A primeira difere
porque usa a matriz robusta — e é a única em que se deve confiar.

## 4. Por que o $F$ homocedástico não serve

Geramos 2.000 amostras em que a nula é **verdadeira** e os erros são fortemente
heterocedásticos, e contamos com que frequência cada teste rejeita a 5%.

In [ ]:
set.seed(760)
M <- 2000; n <- 120
rej_hom <- rej_rob <- logical(M)

for (m in seq_len(M)) {
  x1 <- rnorm(n)
  x2 <- 0.5 * x1 + rnorm(n, 0, sqrt(1 - 0.25))
  y  <- 5 + rnorm(n, 0, 1 + 2 * abs(x1))   # H0 VERDADEIRA; erro heterocedástico
  fit <- lm(y ~ x1 + x2, data = data.frame(y, x1, x2))
  Rm  <- rbind(c(0, 1, 0), c(0, 0, 1))
  rej_rob[m] <- teste_F(fit, Rm, robusto = TRUE)$p  < 0.05
  rej_hom[m] <- teste_F(fit, Rm, robusto = FALSE)$p < 0.05
}

c(nominal = 5,
  F_homocedastico = 100 * mean(rej_hom),
  F_robusto       = 100 * mean(rej_rob))

Contundente: o teste que promete 5% de falsos positivos entrega **mais de 15%** — ele
rejeita hipóteses verdadeiras três vezes mais do que anuncia. A versão robusta fica
perto dos 5% (a folga é aproximação assintótica, e encolhe com $n$).

> **O erro-padrão errado não vicia a estimativa — vicia a conclusão.**

## 5. Por que não testar um coeficiente de cada vez

In [ ]:
set.seed(913)
M <- 4000; n <- 200
rej_F <- rej_qualquer_t <- logical(M)

for (m in seq_len(M)) {
  x1 <- rnorm(n)
  x2 <- 0.3 * x1 + rnorm(n, 0, sqrt(1 - 0.09))   # regressores correlacionados
  y  <- 5 + rnorm(n)                              # H0 verdadeira para AMBOS
  fit <- lm(y ~ x1 + x2, data = data.frame(y, x1, x2))
  tt  <- coef(fit)[-1] / ep_robusto(fit)[-1]
  rej_qualquer_t[m] <- any(abs(tt) > 1.96)
  rej_F[m] <- teste_F(fit, rbind(c(0,1,0), c(0,0,1)))$p < 0.05
}

c(nominal = 5,
  regra_algum_t_rejeita = 100 * mean(rej_qualquer_t),
  teste_F_conjunto      = 100 * mean(rej_F))

A regra ingênua rejeita cerca de **11%** das vezes quando deveria rejeitar 5%. O teste
$F$ fica bem mais perto do alvo. A correção pela correlação entre as estatísticas $t$
não é detalhe técnico: é o que faz o teste ter o tamanho anunciado.

## 6. Restrição única com dois coeficientes

$H_0: \beta_{STR} = \beta_{PctEL}$ — uma restrição, dois coeficientes. Duas
estratégias equivalentes.

In [ ]:
# Estratégia 1: teste F direto, com R = (0, 1, -1)
f_dir <- teste_F(m2, matrix(c(0, 1, -1), nrow = 1))

# Estratégia 2: reparametrização. theta = b_STR - b_PctEL, e o modelo vira
#   Y = b0 + theta*STR + b_PctEL*(STR + PctEL) + u
ca$soma <- ca$STR + ca$PctEL
m_aux   <- lm(TestScore ~ STR + soma, data = ca)
theta   <- unname(coef(m_aux)[2])
ep_th   <- unname(ep_robusto(m_aux)[2])

c(F_direto = f_dir$F, theta = theta, ep_theta = ep_th,
  t_theta = theta / ep_th, t_ao_quadrado = (theta / ep_th)^2,
  p = f_dir$p) |> round(5)

De novo a coincidência exata. E a reparametrização entrega de brinde algo que o teste
$F$ não dá: a **estimativa** da diferença e seu erro-padrão, com os quais se constrói
um intervalo de confiança para ela.

## 7. Especificação: a estabilidade dos coeficientes

O critério para escolher regressores **não** é o $R^2$ — é o exame da estabilidade do
coeficiente de interesse entre especificações alternativas.

In [ ]:
especificacoes <- list(
  `(1) só STR`            = TestScore ~ STR,
  `(2) + PctEL`           = TestScore ~ STR + PctEL,
  `(3) + Expn`            = TestScore ~ STR + PctEL + Expn
)

tab <- t(sapply(especificacoes, function(f) {
  fit <- lm(f, data = ca)
  c(b_STR = unname(coef(fit)[2]),
    ep    = unname(ep_robusto(fit)[2]),
    R2    = summary(fit)$r.squared,
    R2aj  = summary(fit)$adj.r.squared)
}))
round(cbind(tab, verdadeiro = b1), 4)

O coeficiente do $STR$ se move de $-2{,}5$ para $-0{,}9$ à medida que os confundidores
entram — e é **a coluna (3)** que está próxima da verdade ($-1{,}10$), não a de maior
$R^2$ isoladamente. Instabilidade entre especificações é indício de viés remanescente;
foi por isso que a aula insistiu que o $R^2$ é mau conselheiro para especificação.

## Para experimentar

1. Na seção 4, zere a heterocedasticidade (`rnorm(n, 0, 2)`): as duas versões do $F$
   voltam a ~5%. O teste robusto **não custa nada** quando não é preciso.
2. Na seção 5, suba a correlação entre $x_1$ e $x_2$ para 0,9: o tamanho da regra
   ingênua piora ou melhora? Por quê?
3. Na seção 5, passe para **cinco** regressores: o problema de comparações múltiplas
   fica bem mais grave.

---

⬅️ [Aula 5](https://colab.research.google.com/github/vitorwilher/doutorado-epge/blob/main/labs/econometria/05-regressao-multipla.ipynb) · [🏠 Índice](https://colab.research.google.com/github/vitorwilher/doutorado-epge/blob/main/labs/econometria/00-indice.ipynb)

**Fim da sequência.** As seis aulas cobrem os capítulos 1 a 7 de Stock & Watson (2020).